# SOC L2 Lab — Generated Security Events

Queries benign `SOC-L2-Pxx` marked activity produced by `Invoke-SocLabLogGeneration.ps1` from Python.
Designed to run on the Azure ML compute instance inside **Microsoft Sentinel > Notebooks** or locally in VS Code.

> Requirements: `pip install azure-monitor-query azure-identity pandas`. The caller identity requires *Log Analytics Reader* on the workspace.


In [ ]:
# Install required Python packages in Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


In [ ]:
# Lab parameters. Replace WORKSPACE_ID if targeting a different Log Analytics workspace.
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'  # customerId of soc-l2-law
LOOKBACK_HOURS = 24

In [ ]:
from datetime import timedelta
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

def run_kql(query):
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=LOOKBACK_HOURS))
    if resp.status == LogsQueryStatus.PARTIAL:
        table = resp.partial_data[0]
    elif resp.status == LogsQueryStatus.SUCCESS:
        table = resp.tables[0]
    else:
        raise RuntimeError(resp)
    return pd.DataFrame(data=table.rows, columns=table.columns)

## Marker Summary by Table


In [ ]:
summary = '''
let win = SecurityEvent | where EventID in (4625,4688,4720,4722,4724,4725,4726,4727,4728,4729,4730,4732,4733,4738,4776) and (CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2-p02' or TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15') | summarize Events=count() by Table='SecurityEvent';
let sys = Syslog | where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd','su') and SyslogMessage has 'socp04') or (Facility=='local5' and SyslogMessage has 'soc_l2_p13') or (Facility=='local6' and SyslogMessage has 'SOC-L2-P11') | summarize Events=count() by Table='Syslog';
let app = AppServiceHTTPLogs | where UserAgent startswith 'SOC-L2-Lab' | summarize Events=count() by Table='AppServiceHTTPLogs';
let kv = AZKVAuditLogs | where RequestUri has 'soc-l2-p07' or OperationName has 'Secret' | summarize Events=count() by Table='AZKVAuditLogs';
let aks = AKSAuditAdmin | where RequestUri has 'soc-l2-p08' | summarize Events=count() by Table='AKSAuditAdmin';
union win, sys, app, kv, aks | sort by Events desc
'''
run_kql(summary)

## P02 & P15 — Windows Authentication and Account Management


In [ ]:
run_kql('''
SecurityEvent
| where EventID == 4625 and TargetUserName startswith 'soc-l2-p02'
| summarize Attempts=count() by Computer, TargetUserName, LogonType, IpAddress
| sort by Attempts desc
''')

In [ ]:
run_kql('''
SecurityEvent
| where EventID in (4720,4722,4724,4725,4726,4727,4728,4729,4730,4732,4733,4738)
| where TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15'
| project TimeGenerated, Computer, EventID, Activity, SubjectUserName, TargetUserName, MemberName
| sort by TimeGenerated asc
''')

## P04 / P04B — Linux Syslog and Failed SSH/su Attempts


In [ ]:
run_kql('''
Syslog
| where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd','su') and SyslogMessage has 'socp04')
| project TimeGenerated, Computer, Facility, ProcessName, SyslogMessage
| sort by TimeGenerated desc
''')

## P05 / P07 / P08 — App Service, Key Vault, and AKS


In [ ]:
run_kql('''
AppServiceHTTPLogs
| where UserAgent startswith 'SOC-L2-Lab'
| summarize Requests=count() by CsMethod, ScStatus, CsUriStem, CIp
| sort by Requests desc
''')

In [ ]:
run_kql('''
AKSAuditAdmin
| where RequestUri has 'soc-l2-p08'
| project TimeGenerated, Verb, RequestUri, Username=tostring(parse_json(tostring(User)).username)
| sort by TimeGenerated desc
''')